In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import tarfile
import pandas as pd
import numpy as np
import duckdb
import os
import sys
sys.path.append(str(Path(os.getcwd()).parent.absolute()))
import colorlog
logger = colorlog.getLogger()

from ringer_zero.root import read_ttree_as_pdf
TTREE_NAME = 'run_410000/HLT/EgammaMon/summary/events'

In [2]:
source_files = {
    'jf17': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/bkg/user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_without_sigma_constraint_31.10.23_v0_EXT0.tap_jf17_5M_XYZ.root.tgz'
        ),
        'target': 0
    },
    'zee': {
        'datapath': Path(
            '/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/signal/user.isilvafe.mc21_13p6TeV.Zee.AOD.r14136_2sigma_constraint_26.10.23_v0_EXT0.tap_zee_5M_XYZ.root.tgz'
        ),
        'target': 1
    }
}
samples_per_batch = 100000
output_dir = Path('/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet')
if output_dir.exists():
    logger.warning(f'Output directory {output_dir} already exists. Files may be overwritten.')
else:
    output_dir.mkdir(parents=True, exist_ok=True)

 [Sun, 15 Mar 2026 22:46:34] WARNING Output directory /media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet already exists. Files may be overwritten.


In [3]:
def is_safe_int8_cast(series: pd.Series):
    unique_values = series.unique().to_numpy()
    if np.any(np.logical_and(unique_values != 0, unique_values != 1)):
        raise ValueError(f"Series must contain only 1's and 0's for int8 casting, found {unique_values}")
    return True


INT8_CASTS = [
    'el_lhtight',
    'el_lhmedium',
    'el_lhloose',
    'el_lhvloose',
    'el_dnntight',
    'el_dnnmedium',
    'el_dnnloose',
    'mc_hasMC',
    'mc_isTruthElectronAny',
    'mc_isTruthElectronFromZ',
    'mc_isTruthElectronFromJpsiPrompt'
]

def dump_df(df, file_count, processed_rows, output_dir):
    logger.info(f'Processing Batch {file_count} (Rows {processed_rows} to {processed_rows + len(df)})')
    output_path = output_dir / f'data_{file_count:04d}.parquet'
    df['id'] = pd.Series(range(processed_rows, processed_rows + len(df)), dtype="uint64[pyarrow]")
    for col in INT8_CASTS:
        is_safe_int8_cast(df[col])
        df[col] = df[col].astype('bool[pyarrow]')
    df.to_parquet(output_path, index=False)
    return file_count + 1, processed_rows + len(df)

In [4]:
df_accumulator = pd.DataFrame()
file_count = 0
processed_rows = 0

for sample_name, sample_info in source_files.items():
    for root_tar in sample_info['datapath'].glob('*.root.tgz'):
        logger.info(f'Reading {root_tar} for sample {sample_name}')
        with TemporaryDirectory() as temp_dir:
            temp_dir_path = Path(temp_dir)
            with tarfile.open(root_tar, 'r:gz') as tar:
                tar.extractall(path=temp_dir_path)
            for root_file in temp_dir_path.glob('*.root'):
                logger.info(f'Reading {root_file} for sample {sample_name}')
                df = read_ttree_as_pdf(root_file, ttree_name='run_410000/HLT/EgammaMon/summary/events')
                df['target'] = sample_info['target']
                df['target'] = df['target'].astype('uint8[pyarrow]')
                df_accumulator = pd.concat([df_accumulator, df], ignore_index=True)
                if len(df_accumulator) >= samples_per_batch:
                    file_count, processed_rows = dump_df(
                        df_accumulator, file_count, processed_rows, output_dir
                    )
                    df_accumulator = pd.DataFrame()  # Reset accumulator

if len(df_accumulator):
    file_count, processed_rows = dump_df(
        df_accumulator, file_count, processed_rows, output_dir
    )

 [Sun, 15 Mar 2026 22:46:36] INFO Reading /media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/source/bkg/user.isilvafe.mc21_13p6TeV.JF17.AOD.r14136_without_sigma_constraint_31.10.23_v0_EXT0.tap_jf17_5M_XYZ.root.tgz/user.isilvafe.35450371._000001.XYZ.root.tgz for sample jf17
/tmp/ipykernel_1094136/3714514901.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=temp_dir_path)
 [Sun, 15 Mar 2026 22:46:36] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000002.root for sample jf17
 [Sun, 15 Mar 2026 22:46:36] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000010.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000004.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] INFO Reading /tmp/tmpm7lak3e0/output.35450371._000009.root for sample jf17
 [Sun, 15 Mar 2026 22:46:37] IN

# Validate Dataset

In [5]:
output_path_glob = output_dir / 'data_*.parquet'
print(str(output_path_glob))

/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet/data_*.parquet


## Check Target

In [6]:
target_query = f"""
SELECT
    COALESCE(CAST(target AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY target
ORDER BY target ASC;
"""
print(target_query)


SELECT
    COALESCE(CAST(target AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('/media/lucasbanunes/KINGSTON/data/cern-datasets/mc21_isabela_qt_no_restriction/data.parquet/data_*.parquet')
GROUP BY target
ORDER BY target ASC;



In [7]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(target_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,4988951
1,1,2177107


## Check ID

In [8]:
id_query = f"""
SELECT
    COALESCE(CAST(id AS VARCHAR), 'NULL') AS value,
    COUNT(*) as count
FROM read_parquet('{str(output_path_glob)}')
GROUP BY id
ORDER BY id ASC;
"""

In [12]:
with duckdb.connect(':memory:') as conn:
    df = conn.execute(id_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
# df.value_counts(dropna=False)
df

,value,count
0,0,1
1,1,1
2,2,1
3,3,1
4,4,1
...,...,...
7166053,7166053,1
7166054,7166054,1
7166055,7166055,1
7166056,7166056,1


In [13]:
df[df['count'] != 1]

,value,count
